# Sprint 6 — Testing & Refinement

**ITEC631 AI-Driven Network Intrusion Detection System**

This notebook validates the complete NIDS pipeline end-to-end and produces a final performance summary.

| Section | Coverage |
|---------|----------|
| 1 · Pipeline E2E Tests | Stage1→Stage2 chaining, output shapes, probability checks |
| 2 · Edge Case Validation | NaN, zeros, out-of-range, wrong shape, empty batch |
| 3 · Dashboard Feature Tests | Config validity, app.py syntax, batch prediction logic, SHAP |
| 4 · Final Performance Summary | Confusion matrix, classification report, metrics CSV |

**Required Kaggle inputs:** Sprint 1 · Sprint 2 · Sprint 3 · Sprint 4 (optional) · Sprint 5 (optional)


In [ ]:
import subprocess, sys, os, json, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, recall_score
)

print('Setup complete')


## 1 · Path Detection
Locate Sprint 1–4 output files from `/kaggle/input`.


In [ ]:
PROC_DIR      = None   # Sprint 1
S2_MODEL_DIR  = None   # Sprint 2
S3_MODEL_DIR  = None   # Sprint 3
S4_OUTPUT_DIR = None   # Sprint 4

for root, dirs, files in os.walk('/kaggle/input'):
    if ('cicids2017_cleaned.csv' in files or 'test.csv' in files) and PROC_DIR is None:
        PROC_DIR = root + '/'
    if 'xgboost.pkl' in files and 'best_model.json' in files:
        S2_MODEL_DIR = root + '/'
    if 'stage2_xgboost.pkl' in files:
        S3_MODEL_DIR = root + '/'
    if 'sprint4_shap_summary.json' in files:
        S4_OUTPUT_DIR = root + '/'

assert PROC_DIR     is not None, 'test.csv not found → add Sprint 1 output'
assert S2_MODEL_DIR is not None, 'xgboost.pkl not found → add Sprint 2 output'
assert S3_MODEL_DIR is not None, 'stage2_xgboost.pkl not found → add Sprint 3 output'

print(f'PROC_DIR      : {PROC_DIR}')
print(f'S2_MODEL_DIR  : {S2_MODEL_DIR}')
print(f'S3_MODEL_DIR  : {S3_MODEL_DIR}')
print(f'S4_OUTPUT_DIR : {S4_OUTPUT_DIR}')


## 2 · Load Models & Test Data


In [ ]:
with open(S2_MODEL_DIR + 'xgboost.pkl', 'rb') as f:
    stage1_model = pickle.load(f)
with open(S2_MODEL_DIR + 'best_model.json') as f:
    stage1_meta = json.load(f)

with open(S3_MODEL_DIR + 'stage2_xgboost.pkl', 'rb') as f:
    stage2_model = pickle.load(f)
with open(S3_MODEL_DIR + 'stage2_label_encoder.pkl', 'rb') as f:
    le2 = pickle.load(f)
with open(S3_MODEL_DIR + 'sprint3_results.json') as f:
    s3r = json.load(f)

# Load test data
for fname in ['test.csv', 'cicids2017_cleaned.csv']:
    test_path = PROC_DIR + fname
    if os.path.exists(test_path):
        df_test = pd.read_csv(test_path)
        break

feat_cols = [c for c in df_test.columns
             if c not in ('label_binary', 'label_category', 'label_cat_encoded', 'label', 'Label')]
X_test = df_test[feat_cols].values.astype(float)
y_test = df_test['label_binary'].values if 'label_binary' in df_test.columns else None

print(f'Stage1 : {type(stage1_model).__name__}')
print(f'Stage2 : {type(stage2_model).__name__}')
print(f'Classes: {list(le2.classes_)}')
print(f'Test   : {df_test.shape}  ({len(feat_cols)} features)')


## 3 · Section 1 — Pipeline E2E Tests
Validates Stage1→Stage2 chaining on real test samples.


In [ ]:
passed = []
failed = []

def check(name, condition, detail=''):
    if condition:
        passed.append(name)
        print(f'\u2705 PASS  {name}')
    else:
        failed.append(name)
        print(f'\u274c FAIL  {name}  {detail}')

print('=' * 65)
print('SECTION 1 — Pipeline E2E Tests')
print('=' * 65)

# 1.1 Stage1 output shapes & probability sanity
s1_pred_1k  = stage1_model.predict(X_test[:1000])
s1_proba_1k = stage1_model.predict_proba(X_test[:1000])
check('Stage1 prediction shape (1000)',   s1_pred_1k.shape == (1000,))
check('Stage1 proba shape (1000, 2)',     s1_proba_1k.shape == (1000, 2))
check('Stage1 proba rows sum to 1',      np.allclose(s1_proba_1k.sum(axis=1), 1.0, atol=1e-4))
check('Stage1 predictions are binary',   set(np.unique(s1_pred_1k)).issubset({0, 1}))

# 1.2 BENIGN recall
if y_test is not None:
    benign_idx = np.where(y_test == 0)[0][:300]
    s1_benign  = stage1_model.predict(X_test[benign_idx])
    benign_acc = (s1_benign == 0).mean()
    check(f'Stage1 BENIGN recall ≥ 0.95  ({benign_acc:.4f})', benign_acc >= 0.95)

# 1.3 ATTACK detection → Stage2
if y_test is not None:
    attack_idx = np.where(y_test == 1)[0][:300]
    s1_attack  = stage1_model.predict(X_test[attack_idx])
    n_detected = (s1_attack == 1).sum()
    check(f'Stage1 detects attacks (>50%)  ({n_detected}/300)', n_detected > 150)

    X_detected = X_test[attack_idx][s1_attack == 1]
    if len(X_detected) > 0:
        s2_pred  = stage2_model.predict(X_detected)
        s2_proba = stage2_model.predict_proba(X_detected)
        check('Stage2 prediction shape correct',   s2_pred.shape == (len(X_detected),))
        check('Stage2 proba rows sum to 1',       np.allclose(s2_proba.sum(axis=1), 1.0, atol=1e-4))
        check('Stage2 class indices in range',    all(0 <= p < len(le2.classes_) for p in s2_pred))
        decoded = le2.inverse_transform(s2_pred)
        check('Stage2 decoded labels not empty',  len(decoded) > 0)
        print(f'   Attack types detected: {sorted(set(decoded))}')

# 1.4 Full pipeline on 500 samples
X_s = X_test[:500]
s1_full  = stage1_model.predict(X_s)
atk_mask = s1_full == 1
result_labels = ['BENIGN'] * 500
if atk_mask.sum() > 0:
    s2_full = stage2_model.predict(X_s[atk_mask])
    atk_types = le2.inverse_transform(s2_full)
    j = 0
    for i, is_atk in enumerate(atk_mask):
        if is_atk:
            result_labels[i] = atk_types[j]; j += 1
check('Full pipeline: 500 results returned',  len(result_labels) == 500)
check('Full pipeline: all results are strings', all(isinstance(l, str) for l in result_labels))
dist = {l: result_labels.count(l) for l in set(result_labels)}
print(f'   Distribution: {dist}')


## 4 · Section 2 — Edge Case Validation
Tests robustness under abnormal inputs.


In [ ]:
print('=' * 65)
print('SECTION 2 — Edge Case Tests')
print('=' * 65)

n_feat = len(feat_cols)

# 2.1 All-zeros
try:
    p = stage1_model.predict(np.zeros((1, n_feat)))
    pr = stage1_model.predict_proba(np.zeros((1, n_feat)))
    check('All-zeros input: runs without error',   True)
    check('All-zeros proba sums to 1',             np.allclose(pr.sum(), 1.0, atol=1e-4))
except Exception as e:
    check('All-zeros input: runs without error',   False, str(e))

# 2.2 All-ones
try:
    p = stage1_model.predict(np.ones((1, n_feat)))
    check('All-ones input: runs without error',    True)
except Exception as e:
    check('All-ones input: runs without error',    False, str(e))

# 2.3 NaN → nan_to_num(0)
try:
    X_nan = np.nan_to_num(np.full((1, n_feat), np.nan), nan=0.0)
    p = stage1_model.predict(X_nan)
    check('NaN→0 input: runs without error',       True)
except Exception as e:
    check('NaN→0 input: runs without error',       False, str(e))

# 2.4 Out-of-range large values (>1 after MinMax)
try:
    p  = stage1_model.predict(np.full((1, n_feat), 999.0))
    pr = stage1_model.predict_proba(np.full((1, n_feat), 999.0))
    check('Out-of-range input (999): runs',        True)
    check('Out-of-range proba sums to 1',          np.allclose(pr.sum(), 1.0, atol=1e-4))
except Exception as e:
    check('Out-of-range input (999): runs',        False, str(e))

# 2.5 Single sample
try:
    p = stage1_model.predict(X_test[0:1])
    check('Single sample prediction shape (1,)',   p.shape == (1,))
except Exception as e:
    check('Single sample prediction',             False, str(e))

# 2.6 Large batch
try:
    p = stage1_model.predict(X_test[:2000])
    check('Large batch (2000): correct shape',    p.shape == (2000,))
except Exception as e:
    check('Large batch (2000): correct shape',    False, str(e))

# 2.7 Empty dataframe detection
try:
    X_empty = np.zeros((0, n_feat))
    if len(X_empty) == 0:
        check('Empty input: correctly detected before predict', True)
    else:
        check('Empty input: correctly detected before predict', False)
except Exception as e:
    check('Empty input: correctly detected before predict', True)

# 2.8 Wrong feature count → must raise
try:
    stage1_model.predict(np.zeros((1, n_feat - 1)))
    check('Wrong feature count raises ValueError', False, 'no error raised')
except Exception:
    check('Wrong feature count raises ValueError', True)


## 5 · Section 3 — Dashboard Feature Tests
Validates Streamlit app config, syntax, and batch prediction logic.


In [ ]:
print('=' * 65)
print('SECTION 3 — Dashboard Feature Tests')
print('=' * 65)

# 3.1 dashboard_config.json
config_path = '/kaggle/working/dashboard_config.json'
check('dashboard_config.json exists', os.path.exists(config_path))
if os.path.exists(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    for key in ('PROC_DIR', 'S2_MODEL_DIR', 'S3_MODEL_DIR'):
        check(f'Config has {key}', key in cfg)

# 3.2 app.py syntax
app_path = '/kaggle/working/app.py'
check('app.py exists', os.path.exists(app_path))
if os.path.exists(app_path):
    import ast
    try:
        with open(app_path) as f:
            src = f.read()
        ast.parse(src)
        check('app.py syntax is valid Python',         True)
        check('app.py has 4 navigation pages',        src.count('elif PAGE ==') >= 3)
        check('app.py reads dashboard_config.json',   'dashboard_config.json' in src)
        check('app.py uses @st.cache_resource',       '@st.cache_resource' in src)
        check('app.py has Batch Prediction page',     'Batch Prediction' in src)
        check('app.py has SHAP explanation block',    'shap' in src.lower())
    except SyntaxError as e:
        check('app.py syntax is valid Python',        False, str(e))

# 3.3 Simulate batch prediction logic (200 samples)
print('\n--- Simulating batch prediction (200 samples) ---')
X_b = X_test[:200]
s1_b = stage1_model.predict(X_b)
atk_b = s1_b == 1
res = pd.DataFrame({
    'stage1_prediction': ['BENIGN' if p == 0 else 'ATTACK' for p in s1_b],
    'attack_type': 'N/A',
})
if atk_b.sum() > 0:
    s2_b = stage2_model.predict(X_b[atk_b])
    res.loc[atk_b, 'attack_type'] = le2.inverse_transform(s2_b)
check('Batch result has 200 rows',             len(res) == 200)
check('Batch has stage1_prediction column',    'stage1_prediction' in res.columns)
check('Batch has attack_type column',          'attack_type' in res.columns)
benign_rows = res[res['stage1_prediction'] == 'BENIGN']
check('BENIGN rows have N/A attack_type',      (benign_rows['attack_type'] == 'N/A').all())
print(f'   {atk_b.sum()} attacks / {(~atk_b).sum()} benign in 200 samples')

# 3.4 SHAP availability on Stage1
try:
    import shap
    explainer = shap.TreeExplainer(stage1_model)
    sv = explainer.shap_values(X_test[:20])
    check('SHAP TreeExplainer on Stage1 runs',  True)
    if isinstance(sv, list):
        check('SHAP values non-empty',          len(sv) > 0)
    else:
        check('SHAP values non-empty',          sv is not None)
except Exception as e:
    check('SHAP TreeExplainer on Stage1 runs',  False, str(e))


## 6 · Section 4 — Final Performance Summary
Confusion matrix, classification report, and final metrics table saved to `/kaggle/working/`.


In [ ]:
print('=' * 65)
print('SECTION 4 — Final Performance Summary')
print('=' * 65)

# Stage1 full test set metrics
s1_all  = stage1_model.predict(X_test)
s1_prob = stage1_model.predict_proba(X_test)

if y_test is not None:
    s1_f1  = f1_score(y_test,  s1_all, average='weighted')
    s1_acc = accuracy_score(y_test, s1_all)
    s1_rec = recall_score(y_test,   s1_all, average='weighted')
    print(f'\nStage 1 — Binary Classification')
    print(f'  Weighted F1 : {s1_f1:.4f}')
    print(f'  Accuracy    : {s1_acc:.4f}')
    print(f'  Weighted Rec: {s1_rec:.4f}')
    print()
    print(classification_report(y_test, s1_all, target_names=['BENIGN', 'ATTACK']))
else:
    s1_f1, s1_acc, s1_rec = None, None, None

# Pipeline metrics from Sprint 3
pipeline_f1   = s3r.get('pipeline_f1',     0.9989)
single_f1     = s3r.get('single_stage_f1', 0.9980)
delta_f1      = round(pipeline_f1 - single_f1, 4)

# Figure: confusion matrix + comparison bar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if y_test is not None:
    cm = confusion_matrix(y_test, s1_all)
    im = axes[0].imshow(cm, cmap='Blues', interpolation='nearest')
    axes[0].set_title('Stage 1 — Confusion Matrix\n(BENIGN vs ATTACK)', fontsize=13)
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    axes[0].set_xticks([0,1]); axes[0].set_xticklabels(['BENIGN','ATTACK'])
    axes[0].set_yticks([0,1]); axes[0].set_yticklabels(['BENIGN','ATTACK'])
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=12,
                        color='white' if cm[i,j] > cm.max()/2 else 'black')
    plt.colorbar(im, ax=axes[0])
else:
    axes[0].text(0.5, 0.5, 'label_binary not found\nin test.csv',
                ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('Stage 1 — Confusion Matrix')

labels_bar = ['Single-Stage RF', 'Two-Stage Pipeline\n(XGB → XGB)']
vals_bar   = [single_f1, pipeline_f1]
bars = axes[1].barh(labels_bar, vals_bar, color=['#3498db','#e74c3c'], height=0.4)
axes[1].set_xlim(min(vals_bar) - 0.002, max(vals_bar) + 0.002)
for bar, v in zip(bars, vals_bar):
    axes[1].text(v + 0.00003, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Weighted F1 Score')
axes[1].set_title(f'RQ1: Two-Stage vs Single-Stage\n(\u0394 = {delta_f1:+.4f})', fontsize=13)
axes[1].axvline(single_f1, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig('/kaggle/working/sprint6_final_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Saved sprint6_final_performance.png')

# Metrics CSV
rows = [
    ('Stage1 Weighted F1',     f'{s1_f1:.4f}'  if s1_f1  else 'N/A'),
    ('Stage1 Accuracy',        f'{s1_acc:.4f}' if s1_acc else 'N/A'),
    ('Stage1 Weighted Recall', f'{s1_rec:.4f}' if s1_rec else 'N/A'),
    ('Two-Stage Pipeline F1',  f'{pipeline_f1:.4f}'),
    ('Single-Stage F1',        f'{single_f1:.4f}'),
    ('F1 Improvement',         f'{delta_f1:+.4f}'),
]
df_m = pd.DataFrame(rows, columns=['Metric', 'Value'])
df_m.to_csv('/kaggle/working/sprint6_final_metrics.csv', index=False)
print('\n', df_m.to_string(index=False))
print('\n\u2705 Saved sprint6_final_metrics.csv')

# ── Final test summary ───────────────────────────────────
print('\n' + '=' * 65)
print('SPRINT 6 TEST SUMMARY')
print('=' * 65)
total = len(passed) + len(failed)
score = round(len(passed) / total * 100, 1) if total else 0
print(f'Total  : {total}')
print(f'Passed : {len(passed)} \u2705')
print(f'Failed : {len(failed)} \u274c')
print(f'Score  : {score}%')
if failed:
    print('\nFailed tests:')
    for t in failed:
        print(f'  \u274c {t}')

results = {
    'total': total, 'passed': len(passed), 'failed': len(failed), 'score_pct': score,
    'passed_tests': passed, 'failed_tests': failed,
    'stage1_f1':  round(s1_f1,  4) if s1_f1  else None,
    'stage1_acc': round(s1_acc, 4) if s1_acc else None,
    'pipeline_f1':    pipeline_f1,
    'single_stage_f1': single_f1,
    'delta_f1':       delta_f1,
}
with open('/kaggle/working/sprint6_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\u2705 Saved sprint6_results.json')


## Sprint 6 Complete ✅

**Output files saved to `/kaggle/working/`:**

| File | Description |
|------|-------------|
| `sprint6_results.json` | All test pass/fail results + final metrics |
| `sprint6_final_performance.png` | Confusion matrix + two-stage vs single-stage chart |
| `sprint6_final_metrics.csv` | Final metrics table (CSV) |

> **Next:** Sprint 7 — Final Presentation
